## Problem-1. Compare the following two Python codes, which sum all elments in a big vector. 

In [1]:
# Version 1
import array
import time
arr = array.array('l', range(100000000))
sum = 0

begin = time.time()
for i in range(10000):
    for j in range(10000):
        sum += arr[i * 10000 + j]
print(sum)
print(time.time() - begin)

4999999950000000
6.802966356277466


In [2]:
# Version 2
import array
import time
arr = array.array('l', range(100000000))
sum = 0

begin = time.time()
for i in range(10000):
    for j in range(10000):
        sum += arr[j * 10000 + i]
print(sum)
print(time.time() - begin)

4999999950000000
9.112545251846313


Question-1 (10 points): Which code use cache better? Why? Please test the execution time of the two code versions, which code runs faster? Does this result match your original thought?

<strong> Answer: Version 1 uses the cache better because it multiplies the index from the outer loop which results in less multiplication operations being executed altogether. Version 2 uses the index from the inner loop to perform multiplication, causing larger numbers to be multiplied more frequently. After running both versions, the result matches this thought.</strong>

Question-2 (10 points): Does Python utilize CPU cache well? You can investigate this topic by reading online discussion using google.

<strong> Answer: Python does not utilize CPU cache well by default. This is because it does not give allow for direct management with the cache, but offers techniques involving decorators like `@lru_cache` to be added above functions to reduce repeated CPU calculations. </strong>

Question-3 (10 points): A much more efficient way is to use "NumPy". Please use NumPy to do the same computation. What's the execution time? Why does it run much faster?

Hint: You may consider to use "sum" of "NumPy" to do it, such as arr[i*10000:i*10000+10000].sum()

<strong> Answer: (code shown in the cell below) The execution time is 0.0011639595031738281. The reason why the difference in execution time is so substantial is because Numpy utilizes C and Fortran libraries, languages that are leagues more efficient at CPU cache management. </strong>

In [3]:
import numpy as np
import array
import time

arr = array.array('l', range(100000))
sum = 0
i = 0

begin = time.time()
sum = np.sum(arr[i*10000:i*10000+10000])
print(sum)
print(time.time() - begin)

49995000
0.0011639595031738281


## Problem-2. (30 points) Implement a parallel matrix-vector multiplication in Python using MPI. 
You may generate a randon matrix and a corresponding vector, then broadcast the vector and scatter the matrix to all processes, and finally gather the result back.  Please test your code on ARCC cluster (stokes.ist.ucf.edu). Please submit a screen shot for the successful execution of your program. 

See corresponding scripts:

hw1_prob2.py and myjob.slurm

## Problem-3.  Parallel k-Means Clustering (40 points)

Dataset: MNIST digits (subset 10,000 images)  (or sklearn.datasets.load_digits).  

Tasks:

In [29]:
# 1. Implement k-means clustering (sequential code). k=10.  (10 points)
import numpy as np
from sklearn.datasets import load_digits
import time

digits = load_digits()
data = digits.data
n_samples, n_features = data.shape
k = 10

begin = time.time()

# initialize random centroids
np.random.seed(42)
init_indices = np.random.choice(n_samples, k, replace=False)
centroids = data[init_indices]

max_iters = 100

for iteration in range(max_iters):
    # compute Euclidean distances for all points to assign points to nearest centroids
    distances = np.linalg.norm(data[:, np.newaxis, :] - centroids, axis=2)
    labels = np.argmin(distances, axis=1)
    
    # update centroids based on mean of assigned points
    new_centroids = np.zeros_like(centroids)
    for i in range(k):
        cluster_points = data[labels == i]
        if len(cluster_points) > 0:
            new_centroids[i] = np.mean(cluster_points, axis=0)
        else:
            new_centroids[i] = centroids[i] # Keep old if cluster is empty
            
    # check for convergence
    if np.allclose(centroids, new_centroids):
        break
    centroids = new_centroids

print(f"Sequential code runtime: {time.time() - begin:.4f} seconds")

Sequential code runtime: 0.1048 seconds


2. Parallelize both E-step (assignment) and M-step (update) using mpi4py. (20 points)

See hw1_prob3.py script attached

3. Answer the following questions: (10 points)
(a) Does increasing the number of processes in mpi4py make the program run faster?

Increasing the number of processes in mpi4py only increases the amount of time it takes for the program to finish. The times it took for 4, 6, 8, and 10 processes: 0.0155, 0.0161, 0.0273, 0.0206.

(b) Does changing the number of processes have any effect on the accuracy of the K-means clustering results?

To verify the accuracy of the clustering results, the final centroids of the parallelized results were compared to the sequentially-ran code results for num_process = 4, 6, 8, and 10. For each num_process, the parallelized version matched the sequential version.

## Problem-4 (40 points). Please parallelize the following nonlinear regression program using multiprocessing and MPI in python. According to the number of CPU cores in your computer (or one computer on stokes), you may use the same number of processes. 

(0) Initial sequential run

In [22]:
import time
import subprocess

begin = time.time()
subprocess.run(['python', 'NonlinearRegression_Sequential.py'])
print("Time taken for sequential version to finish: ", time.time() - begin)

Figure(640x480)
Final Model Parameters: a=9.502181674800925, b=11.861262607997416, c=10.756680205787243
Figure(640x480)
Time taken for sequential version to finish:  17.54901671409607


(1) MPI in Python (30 points)

In [27]:
begin = time.time()
subprocess.run(['mpirun', '-n', '4', 'python', 'NonlinearRegression_MPI.py'])
print("Time taken for MPI version to finish: ", time.time() - begin)

Final Model Parameters: a=9.502181674800925, b=11.861262607997412, c=10.756680205787253
Figure(640x480)
Time taken for MPI version to finish:  2.4848227500915527


(2) Please compare the entire runtime of the three different implementations (sequential, multiprocessing, MPI) in a report. (10 points)

The main points that can be made about the results of running the three implementations is that the sequential version takes the longest (~17.54 seconds), the MPI version is faster (~2.48 seconds) and the multiprocessing version is the fastest of the three (0.007 seconds). It makes sense that the sequential version takes the longest because it executes the code sequentially, training the model while iterating over the 20,000 datapoints in one process. The MPI version used four processes to distribute work evenly which caused a significant improvement from the sequential version because each of the processes having a smaller amount of data to compute in training the model. The multiprocessing version proved to be the fastest because of its ability to pass messages between CPU cores on the same machine. 

Note: Please see the following scripts attached separately:
NonlinearRegression-Sequential.py, NonlinearRegression-MPI.py, and NonlinearRegression-MulProc.py

In [28]:
begin = time.time()
subprocess.run(['python', 'NonlinearRegression-MulProc.py'])
print("Time taken for Multi processing version to finish: ", time.time() - begin)

Time taken for Multi processing version to finish:  0.007130146026611328


python: can't open file '/home/viviteto/Desktop/school/cloud_computation/hw1/NonlinearRegression-MulProc.py': [Errno 2] No such file or directory
